# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets and their field and column `@id`s. This helps us understand how the dataset is structured.

In [ ]:
# List all available record sets, their fields and columns by their @id

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        # Each record set is a dict
        rs_id = rs.get('@id', '(no @id)')
        print(f"Record Set @id: {rs_id}")
        # Fields
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            field_id = field.get('@id', '(no @id)')
            print(f"    Field @id: {field_id}, Name: {field.get('name', None)}")
        # Columns
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        print("  Columns:")
        for col in columns:
            col_id = col.get('@id', '(no @id)')
            print(f"    Column @id: {col_id}, Name: {col.get('name', None)}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Entities are referenced by their `@id`.

> **Note:** Update the `record_sets_ids` variable with the `@id`s found in the previous step. If there are multiple record sets, you can extract them all.

In [ ]:
# Discover available record set @id(s)
record_sets = getattr(metadata, 'recordSet', [])
record_set_ids = []
for rs in record_sets:
    rs_id = rs.get('@id', None)
    if rs_id:
        record_set_ids.append(rs_id)

# Preview
print(f"Found record sets: {record_set_ids}")

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading DataFrame for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    if not dataframes[record_set_id].empty:
        print(f"Columns for Record Set @id {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for Record Set @id {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps—filter records, normalize numeric fields, group by key attributes. 

Below, choose a numeric field and a group-by field by their `@id` (see previous output for available options).

In [ ]:
# Replace these with real @id names of fields/columns from your dataset!

chosen_record_set_id = record_set_ids[0] if record_set_ids else None

if chosen_record_set_id and not dataframes[chosen_record_set_id].empty:
    df = dataframes[chosen_record_set_id]
    
    # List all column names for reference
    print(f"Columns available: {df.columns.tolist()}")
    
    # For demonstration, choose a likely numeric field (e.g., containing 'MW' or 'Abundance')
    # You need to update `numeric_field_id` and `group_field_id` based on your data's actual column @id entries.
    # You can print columns to decide which to use.
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if 'molecular_weight' in col or 'MW' in col or 'abundance' in col:
            numeric_field_id = col
            break
    for col in df.columns:
        if 'sample' in col or 'Sample' in col:
            group_field_id = col
            break
    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")
    
    # Ensure the numeric field is numeric
    if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        # Filtering records
        threshold = df[numeric_field_id].mean()  # Demo threshold: above mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by some grouping field if present
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found or field is not numeric.")
else:
    print("No available record set or the DataFrame is empty.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of the chosen numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id and not dataframes[chosen_record_set_id].empty and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped data is available, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,6))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("Visualization fields/data not available.")

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to access and analyze the FAIR^2 dataset. You can adapt the filtering/grouping/EDA cells with additional operations for your own scientific use and exploratory analyses.
